In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score, accuracy_score
from sklearn.compose import ColumnTransformer 
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

# Download dataset
url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"

df = pd.read_csv(url)

# Replace missing values
categorical_cols = df.select_dtypes(include=["object", "category"]).columns
numerical_cols = df.select_dtypes(include=["number"]).columns

df[categorical_cols] = df[categorical_cols].fillna("NA")
df[numerical_cols] = df[numerical_cols].fillna(0.0)

# Split data using the exact homework instructions
df_full_train, df_test = train_test_split(
    df, test_size=0.2, random_state=42
)

df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

# Separate target from features
X_train = df_train.drop(columns=["converted"]).copy()
X_val = df_val.drop(columns=["converted"]).copy()

y_train = df_train["converted"].copy()
y_val = df_val["converted"].copy()

print("Dataset loaded successfully!")
print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", df_test.shape)
print("Target values:", y_train.unique())

Dataset loaded successfully!
Training: (3000, 8)
Validation: (1000, 8)
Test: (1000, 9)
Target values: [1 0]


/tmp/ipykernel_11180/1270588918.py:17: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=["object", "category"]).columns


In [2]:
#Q1
print("Most frequent industry:", df["industry"].mode()[0])

Most frequent industry: technology


In [3]:
#Q2
correlation_matrix = df.select_dtypes(include=["number"]).corr()

pairs = [
    ("interaction_count", "lead_score"),
    ("number_of_courses_viewed", "lead_score"),
    ("number_of_courses_viewed", "interaction_count"),
    ("annual_income", "interaction_count")
]

correlations = {
    f"{a} and {b}": correlation_matrix.loc[a, b]
    for a, b in pairs
}

for pair, value in correlations.items():
    print(f"{pair}: {value:.4f}")

print("\nHighest correlation:", max(correlations, key=correlations.get))

interaction_count and lead_score: 0.9157
number_of_courses_viewed and lead_score: 0.7572
number_of_courses_viewed and interaction_count: 0.7216
annual_income and interaction_count: 0.1228

Highest correlation: interaction_count and lead_score


In [5]:
#Q3
categorical_features = X_train.select_dtypes(
    include=["object", "category"]
).columns

mi_scores = {
    col: round(mutual_info_score(X_train[col], y_train), 2)
    for col in categorical_features
}

for feature in ["industry", "location", "lead_source", "employment_status"]:
    print(f"{feature}: {mi_scores[feature]}")

print(
    "\nAnswer:",
    max(
        ["industry", "location", "lead_source", "employment_status"],
        key=lambda feature: mi_scores[feature]
    )
)

industry: 0.0
location: 0.0
lead_source: 0.03
employment_status: 0.02

Answer: lead_source


/tmp/ipykernel_11180/294887137.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X_train.select_dtypes(


In [6]:
#Q4
categorical_columns = X_train.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

numerical_columns = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_columns),
        ("numerical", "passthrough", numerical_columns)
    ]
)

model = make_pipeline(
    preprocessor,
    LogisticRegression(
        solver="liblinear",
        C=1.0,
        max_iter=1000,
        random_state=42
    )
)

model.fit(X_train, y_train)

y_pred = model.predict(X_val)
accuracy = accuracy_score(y_val, y_pred)

print("Validation accuracy:", round(accuracy, 2))

Validation accuracy: 0.65


/tmp/ipykernel_11180/1535675140.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = X_train.select_dtypes(


In [7]:
#Q5
features_to_remove = [
    "lead_source",
    "number_of_courses_viewed",
    "interaction_count"
]

baseline_accuracy = accuracy_score(y_val, model.predict(X_val))
results = {}

for feature in features_to_remove:
    X_train_reduced = X_train.drop(columns=[feature])
    X_val_reduced = X_val.drop(columns=[feature])

    cat_cols = X_train_reduced.select_dtypes(
        include=["object", "category", "bool"]
    ).columns.tolist()

    num_cols = X_train_reduced.select_dtypes(
        include=["number"]
    ).columns.tolist()

    reduced_preprocessor = ColumnTransformer(
        transformers=[
            ("categorical", OneHotEncoder(handle_unknown="ignore"), cat_cols),
            ("numerical", "passthrough", num_cols)
        ]
    )

    reduced_model = make_pipeline(
        reduced_preprocessor,
        LogisticRegression(
            solver="liblinear",
            C=1.0,
            max_iter=1000,
            random_state=42
        )
    )

    reduced_model.fit(X_train_reduced, y_train)

    reduced_accuracy = accuracy_score(
        y_val,
        reduced_model.predict(X_val_reduced)
    )

    results[feature] = {
        "accuracy_without_feature": reduced_accuracy,
        "difference": baseline_accuracy - reduced_accuracy
    }

results_df = pd.DataFrame(results).T
results_df["absolute_difference"] = results_df["difference"].abs()

print(results_df.sort_values("absolute_difference"))

print(
    "\nAnswer:",
    results_df["absolute_difference"].idxmin()
)

                          accuracy_without_feature  difference  \
number_of_courses_viewed                     0.643       0.002   
lead_source                                  0.642       0.003   
interaction_count                            0.601       0.044   

                          absolute_difference  
number_of_courses_viewed                0.002  
lead_source                             0.003  
interaction_count                       0.044  

Answer: number_of_courses_viewed


/tmp/ipykernel_11180/930754820.py:15: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train_reduced.select_dtypes(
/tmp/ipykernel_11180/930754820.py:15: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on 

In [8]:
#Q6
C_values = [0.000001, 0.00001, 0.0001, 0.001]

results = []

for C in C_values:
    model_reg = make_pipeline(
        ColumnTransformer(
            transformers=[
                ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_columns),
                ("numerical", "passthrough", numerical_columns)
            ]
        ),
        LogisticRegression(
            solver="liblinear",
            C=C,
            max_iter=1000,
            random_state=42
        )
    )

    model_reg.fit(X_train, y_train)

    predictions = model_reg.predict(X_val)
    score = accuracy_score(y_val, predictions)

    results.append({
        "C": C,
        "accuracy": score,
        "rounded_accuracy": round(score, 3)
    })

results_df = pd.DataFrame(results)

print(results_df)

best_accuracy = results_df["accuracy"].max()

best_C = results_df.loc[
    results_df["accuracy"] == best_accuracy, "C"
].min()

print("\nBest C:", best_C)
print("Best accuracy:", round(best_accuracy, 3))

          C  accuracy  rounded_accuracy
0  0.000001     0.598             0.598
1  0.000010     0.598             0.598
2  0.000100     0.613             0.613
3  0.001000     0.645             0.645

Best C: 0.001
Best accuracy: 0.645
